<a href="https://colab.research.google.com/github/saisathvik485/BIS-LAB-5M-/blob/main/traffing_signal_optimization_using_pso.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import copy

class TrafficLightParticle:
    """Represents a single candidate traffic light configuration."""
    def __init__(self, num_phases, min_green, max_green):
        self.num_phases = num_phases
        # Position vector: represents green time duration per phase (in seconds)
        self.position = [random.uniform(min_green, max_green) for _ in range(num_phases)]
        # Velocity vector: rate of change for the phase timings
        self.velocity = [0.0] * num_phases

        # Performance memory tracking
        self.pbest_position = copy.deepcopy(self.position)
        self.pbest_fitness = float('inf')

class PSOTrafficOptimizer:
    """Main optimizer engine handling the swarm mechanics."""
    def __init__(self, num_phases=4, swarm_size=20, max_iterations=50):
        self.num_phases = num_phases
        self.swarm_size = swarm_size
        self.max_iterations = max_iterations

        # Operational constraints (Legal phase lengths)
        self.min_green = 10.0   # e.g., safe minimum pedestrian crossing clearance
        self.max_green = 60.0   # Maximum time before starving other approaches

        # PSO Parameters
        self.w = 0.5            # Inertia weight
        self.c1 = 1.5           # Cognitive parameter (personal best pull)
        self.c2 = 1.5           # Social parameter (global best pull)

        # Swarm State
        self.swarm = []
        self.gbest_position = []
        self.gbest_fitness = float('inf')

    def simulate_traffic_delay(self, timings):
        """
        Evaluates the current green splits.
        REPLACE THIS MOCK EQUATION WITH YOUR ACTUAL SIMULATOR INTERFACE (e.g., TraCI/SUMO).
        Goal: Minimize total vehicle delay metric.
        """
        # Mock traffic behavior: phase adjustments closer to target benchmarks yield less delay
        target_ideal_splits = [30.0, 20.0, 40.0, 15.0]
        return sum((t - ideal) ** 2 for t, ideal in zip(timings, target_ideal_splits))

    def _clamp_constraints(self, position):
        """Maintains structural integrity and traffic code boundaries."""
        return [max(self.min_green, min(self.max_green, x)) for x in position]

    def run_optimization(self):
        # 1. INITIALIZE SWARM
        self.swarm = [TrafficLightParticle(self.num_phases, self.min_green, self.max_green) for _ in range(self.swarm_size)]

        for particle in self.swarm:
            particle.position = self._clamp_constraints(particle.position)
            initial_fit = self.simulate_traffic_delay(particle.position)

            particle.pbest_fitness = initial_fit
            particle.pbest_position = copy.deepcopy(particle.position)

            if initial_fit < self.gbest_fitness:
                self.gbest_fitness = initial_fit
                self.gbest_position = copy.deepcopy(particle.position)

        # 2. OPTIMIZATION LOOP
        for it in range(self.max_iterations):
            for particle in self.swarm:
                for j in range(self.num_phases):
                    # Standard PSO Velocity Calculation
                    r1, r2 = random.random(), random.random()
                    cognitive = self.c1 * r1 * (particle.pbest_position[j] - particle.position[j])
                    social = self.c2 * r2 * (self.gbest_position[j] - particle.position[j])

                    particle.velocity[j] = (self.w * particle.velocity[j]) + cognitive + social
                    particle.position[j] += particle.velocity[j]

                # Apply limits & boundaries
                particle.position = self._clamp_constraints(particle.position)

                # Single evaluation checkpoint to prevent excessive computational overhead
                current_fit = self.simulate_traffic_delay(particle.position)

                # Update Best Configurations
                if current_fit < particle.pbest_fitness:
                    particle.pbest_position = copy.deepcopy(particle.position)
                    particle.pbest_fitness = current_fit

                if current_fit < self.gbest_fitness:
                    self.gbest_fitness = current_fit
                    self.gbest_position = copy.deepcopy(particle.position)

            if (it + 1) % 10 == 0:
                print(f"Iteration {it+1}/{self.max_iterations} | Current Minimized Delay Score: {self.gbest_fitness:.2f}")

        return [round(t, 1) for t in self.gbest_position], round(self.gbest_fitness, 2)

# --- EXECUTION ENTRYPOINT ---
if __name__ == "__main__":
    optimizer = PSOTrafficOptimizer(num_phases=4, swarm_size=25, max_iterations=50)
    best_timings, final_delay_score = optimizer.run_optimization()

    print("\n=== OPTIMIZATION RESULT ===")
    print(f"Optimal Green Timings per Phase (seconds): {best_timings}")
    print(f"Final Optimized Network Delay Index       : {final_delay_score}")


Iteration 10/50 | Current Minimized Delay Score: 0.48
Iteration 20/50 | Current Minimized Delay Score: 0.00
Iteration 30/50 | Current Minimized Delay Score: 0.00
Iteration 40/50 | Current Minimized Delay Score: 0.00
Iteration 50/50 | Current Minimized Delay Score: 0.00

=== OPTIMIZATION RESULT ===
Optimal Green Timings per Phase (seconds): [30.0, 20.0, 40.0, 15.0]
Final Optimized Network Delay Index       : 0.0
